# Practical Session: Eddy Covariance Processing & Configuration Comparison

## 1. Introduction and Objectives
This notebook processes and compares atmospheric flux and meteorological outputs derived from different EddyPro configurations (Basic vs. Biomet). 

### Key Objectives:
1. **Time Synchronization**: Align observation timestamps by applying UTC offset adjustments ($+3\text{ hours}$).
2. **Data Regularization**: Resample high-frequency turbulent fluxes and meteorological variables into regular $1\text{-hour}$ averages.
3. **Configuration Comparison (Basic vs. Biomet)**: Evaluate the impact of incorporating auxiliary meteorological measurements—specifically relative humidity (`RH`) from the Biomet setup—on the processed outputs.
4. **Multi-variable Visualization**: Generate synchronized multi-panel time-series plots to contrast turbulent fluxes and meteorological state variables between both setups.

---

In [ ]:
# --- Google Colab Integration --- 
# Uncomment the following lines if you are running in Google Colab to connect your Google Drive
#from google.colab import drive
#drive.mount('/content/drive/')

#%cd /content/drive/MyDrive/cryoextremes-practicals

#!pwd

#!ls

## 2. Environment Setup
Import required standard numerical analysis, file I/O, and data visualization libraries.

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 4. Processing Eddy Covariance (EddyPro) Outputs: Basic vs. Biomet Configurations
Parse high-frequency flux calculation outputs from both EddyPro setups, handle missing value codes (`-9999`), align time indexes, and resample parameters to $1\text{-hour}$ temporal resolution. Note that the **Biomet configuration** integrates supplementary measurements such as relative humidity (`RH`), allowing a direct comparative assessment against the **Basic configuration**.

In [ ]:
# File path to EddyPro Basic configuration output
file_path_basic = "data/AP/eddypro_PG_OP43_full_output_2026-10-07T104038_exp.csv"

# Load output while skipping header metadata and replacing missing indicators
df_basic = pd.read_csv(file_path_basic, sep=",", skiprows=[0, 2], na_values=-9999)

# Select micrometeorological and state variables
cols_eddy = ["date", "time", "H", "LE", "air_temperature", "air_pressure", "RH", "wind_speed", "wind_dir"]
df_sel_basic = df_basic[cols_eddy].copy()

# Construct datetime index with time shift (+3 hours)
df_sel_basic["datetime"] = pd.to_datetime(df_sel_basic["date"] + " " + df_sel_basic["time"]) + pd.Timedelta(hours=3)
df_sel_basic = df_sel_basic.set_index("datetime").drop(columns=["date", "time"])

# Restrict study period and resample to 1-hour averages
df_sel_basic = df_sel_basic.loc["2024-12-06":"2024-12-13"].resample("1h").mean()
df_sel_basic.head()

In [ ]:
# File path to EddyPro Biomet configuration output (incorporates auxiliary RH and biomet data)
file_path_biomet = "data/AP/eddypro_PG_OP43_full_output_2026-10-14T124645_exp.csv"

# Load output while skipping header metadata and replacing missing indicators
df_biomet = pd.read_csv(file_path_biomet, sep=",", skiprows=[0, 2], na_values=-9999)

# Select micrometeorological and state variables
cols_eddy = ["date", "time", "H", "LE", "air_temperature", "air_pressure", "RH", "wind_speed", "wind_dir"]
df_sel_biomet = df_biomet[cols_eddy].copy()

# Construct datetime index with time shift (+3 hours)
df_sel_biomet["datetime"] = pd.to_datetime(df_sel_biomet["date"] + " " + df_sel_biomet["time"]) + pd.Timedelta(hours=3)
df_sel_biomet = df_sel_biomet.set_index("datetime").drop(columns=["date", "time"])

# Restrict study period and resample to 1-hour averages
df_sel_biomet = df_sel_biomet.loc["2024-12-06":"2024-12-13"].resample("1h").mean()
df_sel_biomet.head()

In [ ]:
# Variables to compare between Basic and Biomet configurations (including relative humidity RH)
variables = [
    'H', 'LE', 'air_temperature', 'air_pressure', 
    'RH', 'wind_speed', 'wind_dir'
]

# Create a comparative figure with subplots for each variable sharing the x-axis
fig, axes = plt.subplots(nrows=len(variables), ncols=1, figsize=(12, 8), sharex=True)

# Loop through each variable and its corresponding axis to plot both configurations
for i, var in enumerate(variables):
    ax = axes[i]
    
    # Plot EddyPro Basic configuration
    ax.plot(
        df_sel_basic.index, 
        df_sel_basic[var], 
        label='Basic', 
        color='blue', 
        linewidth=1.5,
        alpha=0.8
    )
    
    # Plot EddyPro Biomet configuration (incorporating auxiliary RH data)
    ax.plot(
        df_sel_biomet.index, 
        df_sel_biomet[var], 
        label='Biomet (with auxiliary RH)', 
        color='orange', 
        linewidth=1.2,
        alpha=0.8,
        linestyle='--'
    )
    
    # Set labels and legend
    ax.set_ylabel(var, fontsize=10)
    ax.legend(loc='best', fontsize=8, ncol=2)
    ax.grid(True, linestyle=':', alpha=0.6)

# Adjust layout and labeling
plt.tight_layout()

# Save and display comparison figure
os.makedirs('fig', exist_ok=True)
plt.savefig('fig/variable_comparison_basic_vs_biomet.png', dpi=300, bbox_inches='tight')
plt.show()